# Which columns carry the same information


**The question (Einar, 2026-11-07):** does the information map use feature correlation to decide which features hold the same information? Until now, no -- categories come from where a column was measured (provenance). This report measures it: rank correlation between every pair of numeric columns, groups of columns that move together (|Spearman| >= 0.8 for every pair in a group), the effective number of independent axes, and where the measured groups cross the declared categories.

Nothing is changed by this report; it is the evidence for deciding whether to add a measured redundancy rule next to the category rule.

Built 2026-11-07.

In [1]:
import os, numpy as np, pandas as pd
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from parascape import information_categories as IC
nodes = pd.read_parquet(os.path.join(REPO, 'parascape/data/nodes.parquet'))
num = nodes.select_dtypes('number')
num = num.loc[:, (num.notna().sum() >= 1000) & (num.nunique() > 2)]
print(num.shape[1], 'numeric columns measured on >= 1,000 genes (binary columns left out)')

322 numeric columns measured on >= 1,000 genes (binary columns left out)


## 1 · Pairwise rank correlation

On the genes both columns measure; pairs sharing fewer than 300 genes are left undecided (not counted as independent).

In [2]:
ranks = num.rank()
rho = ranks.corr(method='pearson', min_periods=300)   # Pearson on ranks = Spearman, pairwise complete
undecided = int(rho.isna().values[np.triu_indices(len(rho), 1)].sum())
pairs = rho.where(np.triu(np.ones(rho.shape, bool), 1)).stack()
strong = pairs[pairs.abs() >= 0.8]
print(len(pairs), 'pairs;', len(strong), 'with |Spearman| >= 0.8;', undecided, 'undecided (too few shared genes)')

103684 pairs; 2733 with |Spearman| >= 0.8; 129 undecided (too few shared genes)


## 2 · Groups of columns that move together

In [3]:
d = (1 - rho.abs()).fillna(1.0).clip(lower=0).to_numpy(copy=True)
np.fill_diagonal(d, 0)
d = (d + d.T) / 2
Z = linkage(squareform(d, checks=False), method='complete')
group = pd.Series(fcluster(Z, t=0.2, criterion='distance'), index=rho.index)
cat = pd.Series({c: '+'.join(IC.categories_of(c)) or 'unannotated' for c in rho.index})
sizes = group.value_counts()
multi = sizes[sizes > 1].index
groups = pd.DataFrame([dict(group=int(g), columns=int(sizes[g]), categories=', '.join(sorted(set(cat[group == g].astype(str)))),
                            example=', '.join(list(group[group == g].index[:5]))) for g in multi]).sort_values('columns', ascending=False)
groups['crosses_categories'] = groups.categories.str.contains(',')
groups.to_csv(os.path.join(OUT, 'groups.tsv'), sep='\t', index=False)
pd.DataFrame({'column': group.index, 'group': group.values, 'category': cat.values}).to_csv(os.path.join(OUT, 'column_groups.tsv'), sep='\t', index=False)
print(len(rho), 'columns fall into', group.nunique(), 'groups;', len(multi), 'groups hold more than one column;',
      int(groups.crosses_categories.sum()), 'of those cross a declared category')
groups.head(25)

322 columns fall into 149 groups; 44 groups hold more than one column; 2 of those cross a declared category


,group,columns,categories,example,crosses_categories
0,141,27,"transcript, translation","rpf99395_extracellular_r1, rpf99395_extracellular_r2, rpf99395_intracellular_r1, rpf99395_intracellular_r2, rpf129869_confluent_r1",True
1,147,20,transcript,"stress_5-1_unstress_red_24, stress_5-2_unstress_red_24, stress_6-1_unstress_red_24, stress_6-4_unstress_red_24, stress_7-1_unstress_red_24",False
2,146,18,transcript,"brain168465_1d_base_mean, brain168465_2d_base_mean, brain168465_4d_base_mean, brain168465_7d_base_mean, brain168465_14d_base_mean",False
3,136,12,transcript,"rna108740_Tachyzoites_T2_FPKM, rna108740_Tachyzoites_T4_FPKM, expr_tachy, invivo_TZ_1, invivo_TZ_2",False
4,112,12,protein_abundance,"proteome_log2_normalized_and_imputated_abundances_UT_R1, proteome_log2_normalized_and_imputated_abundances_UT_R2, proteome_log2_normalized_and_imputated_abundances_UT_R3, proteome_log2_normalized_and_imputated_abundances_T_24h_R1, proteome_log2_normalized_and_imputated_abundances_T_24h_R2",False
5,130,9,transcript,"rna108740_Tissue_cysts_A_FPKM, rna108740_Tissue_cysts_B_FPKM, expr_cyst, invivo_BZ_28DPI_1, invivo_BZ_28DPI_3",False
6,135,8,transcript,"cpsf4rna168155_ut_1, cpsf4rna168155_ut_2, cpsf4rna168155_iaa_7h_1, cpsf4rna168155_iaa_7h_2, cpsf4rna168155_iaa_24h_1",False
7,115,7,transcript,"rna108740_Day3_CS4_FPKM, rna108740_Day3_CS5_FPKM, rna108740_Day3_CS6_FPKM, rna108740_Day5_K4_14_FPKM, rna108740_Day7_CS7_FPKM",False
8,133,6,transcript,"morc_MORC_IAA1, morc_MORC_IAA2, morc_MORC_IAA3, morc_MORC-KD-BFD1-KO_IAA1, morc_MORC-KD-BFD1-KO_IAA2",False
9,134,6,transcript,"morc_DD-BFD1-Ty_UT1, morc_DD-BFD1-Ty_UT2, morc_DD-BFD1-Ty_UT3, morc_DD-BFD1-Ty_Shield1, morc_DD-BFD1-Ty_Shield2",False


## 3 · How many independent axes

Participation ratio of the eigenvalues of the correlation matrix (missing correlations set to 0): (sum of eigenvalues)^2 / sum of squared eigenvalues. It equals the number of columns if all are independent and 1 if all carry one signal.

In [4]:
ev = np.clip(np.linalg.eigvalsh(rho.fillna(0).values), 0, None)
pr = float(ev.sum() ** 2 / (ev ** 2).sum())
axes = dict(columns=len(rho), groups_at_0_8=int(group.nunique()), participation_ratio=round(pr, 1))
pd.Series(axes).to_csv(os.path.join(OUT, 'axes.tsv'), sep='\t', header=False)
axes

{'columns': 322, 'groups_at_0_8': 149, 'participation_ratio': 7.0}

## 4 · Where measured redundancy crosses the declared categories

Each such group is a place where an input from one category carries the label information of another, so the category rule alone would not catch it.

In [5]:
cross = groups[groups.crosses_categories]
cross.to_csv(os.path.join(OUT, 'crossing_groups.tsv'), sep='\t', index=False)
cross

,group,columns,categories,example,crosses_categories
0,141,27,"transcript, translation","rpf99395_extracellular_r1, rpf99395_extracellular_r2, rpf99395_intracellular_r1, rpf99395_intracellular_r2, rpf129869_confluent_r1",True
23,8,2,"conservation, sequence","length, snp_synonymous",True
